In [3]:
"""
Sensitivity Analysis - Fixed & Simplified Version
Generates only 2 publication-quality combined figures
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION
# ============================================
SEED = 2411
N_STEPS = 120
DISRUPTION_AT_STEP = 10
OUTPUT_DIR = "./sensitivity_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    try:
        plt.style.use('seaborn-whitegrid')
    except:
        pass

STRATEGY_COLORS = {
    'baseline': '#E74C3C',
    'dual_only': '#E67E22', 
    'safety_only': '#27AE60',
    'flex_only': '#2ECC71',
    'dynalloc_only': '#F1C40F',
    'all_combined': '#3498DB',
}
STRATEGY_LABELS = {
    'baseline': 'Baseline',
    'dual_only': 'Dual Sourcing',
    'safety_only': 'Safety Stock',
    'flex_only': 'Flexible Capacity',
    'dynalloc_only': 'Dynamic Allocation',
    'all_combined': 'All Combined',
}
STRATEGY_CONFIGS = {
    "baseline": {"dual_sourcing": False, "safety_stock_factor": 1.0, "flexible_capacity": False, "dynamic_reallocation": False},
    "dual_only": {"dual_sourcing": True, "safety_stock_factor": 1.0, "flexible_capacity": False, "dynamic_reallocation": False},
    "safety_only": {"dual_sourcing": False, "safety_stock_factor": 1.3, "flexible_capacity": False, "dynamic_reallocation": False},
    "flex_only": {"dual_sourcing": False, "safety_stock_factor": 1.0, "flexible_capacity": True, "dynamic_reallocation": False},
    "dynalloc_only": {"dual_sourcing": False, "safety_stock_factor": 1.0, "flexible_capacity": False, "dynamic_reallocation": True},
    "all_combined": {"dual_sourcing": True, "safety_stock_factor": 1.2, "flexible_capacity": True, "dynamic_reallocation": True}
}

""" --NOT FIXED YET--
def run_capacity_loss_test(capacity_loss_frac, strategy_config, seed):
    
    #Run capacity_loss scenario with EXPLICIT capacity_loss_frac.
    #The key fix: we override the model's internal value AFTER creation.
    
    random.seed(seed)
    np.random.seed(seed)
    
    from model import MultiTierModel
    
    assumptions = {
        "n_suppliers": 4, "n_plants": 2, "n_dcs": 2, "n_retailers": 4,
        "base_stock": {"supplier": 60, "plant": 50, "dc": 30, "retailer": 20},
        "capacity": {"supplier": 50, "plant": 60, "dc": 0, "retailer": 0},
        "lead_time": {"supplier": 3, "plant": 2, "dc": 2, "retailer": 1},
        "capacity_loss_frac": capacity_loss_frac,  # Set here
        "recovery_duration": 15,
        "holding_cost": 1.0, "backlog_cost": 5.0,
        "retailer_demand_mean": 7.0
    }
    
    model = MultiTierModel(
        assumptions=assumptions,
        seed=seed,
        strategies=strategy_config.copy(),
        disruption_at_step=DISRUPTION_AT_STEP,
        scenario="none",
        n_steps=N_STEPS
    )
    
    # FIX: Explicitly override the capacity_loss_frac in model
    model.capacity_loss_frac = capacity_loss_frac
    
    # Also need to patch the disruption trigger to use our value
    original_trigger = model._maybe_trigger_disruption
    cap_frac = capacity_loss_frac  # Capture in closure
    
    def patched_trigger():
        if (not model.disruption_done) and (model.time == model.disruption_at_step):
            # Find victim
            candidates = [a for a in model.all_agents if a.tier in ("supplier", "plant")]
            if candidates:
                victim = random.choice(candidates)
                model._victim = victim
                tier = victim.tier
                model._affected_agents = [a for a in model.all_agents if a.tier == tier]
                
                for agent in model._affected_agents:
                    agent._orig_capacity = agent.capacity
                    # USE OUR capacity_loss_frac value!
                    agent.capacity = int(agent.capacity * (1 - cap_frac))
                    agent.is_disrupted = True
                    agent.recovery_timer = model.recovery_duration
            
            model.disruption_done = True
            model.disruption_step = model.time
    
    model._maybe_trigger_disruption = patched_trigger
    
    for _ in range(N_STEPS):
        model.step()
    
    df = model.datacollector.get_model_vars_dataframe()
    post_df = df[df.index >= DISRUPTION_AT_STEP]
    
    return {
        "post_fill": post_df["fill_rate"].mean() if len(post_df) > 0 else np.nan,
        "min_fill": post_df["fill_rate"].min() if len(post_df) > 0 else np.nan,
        "final_cost": model.compute_total_cost(),
        "ttr": model.compute_time_to_recover(target_frac=0.95),
    }
"""

def run_demand_spike_test(spike_factor, strategy_config, seed):
    """Run demand_spike scenario with explicit spike factor."""
    random.seed(seed)
    np.random.seed(seed)
    
    from model import MultiTierModel
    
    assumptions = {
        "n_suppliers": 4, "n_plants": 2, "n_dcs": 2, "n_retailers": 4,
        "base_stock": {"supplier": 60, "plant": 50, "dc": 30, "retailer": 20},
        "capacity": {"supplier": 50, "plant": 60, "dc": 0, "retailer": 0},
        "lead_time": {"supplier": 3, "plant": 2, "dc": 2, "retailer": 1},
        "capacity_loss_frac": 0.1, "recovery_duration": 5,
        "holding_cost": 1.0, "backlog_cost": 5.0,
        "retailer_demand_mean": 7.0
    }
    
    model = MultiTierModel(
        assumptions=assumptions,
        seed=seed,
        strategies=strategy_config.copy(),
        disruption_at_step=DISRUPTION_AT_STEP,
        scenario="demand_spike",
        n_steps=N_STEPS
    )
    
    factor = spike_factor
    def patched_trigger():
        if (not model.disruption_done) and (model.time == model.disruption_at_step):
            model.retailer_demand_mean = model._retailer_demand_baseline * factor
            for r in model.retailers:
                r.is_disrupted = True
                r.recovery_timer = model.recovery_duration
            model.disruption_done = True
            model.disruption_step = model.time
    
    model._maybe_trigger_disruption = patched_trigger
    
    for _ in range(N_STEPS):
        model.step()
    
    df = model.datacollector.get_model_vars_dataframe()
    post_df = df[df.index >= DISRUPTION_AT_STEP]
    
    return {
        "post_fill": post_df["fill_rate"].mean() if len(post_df) > 0 else np.nan,
        "min_fill": post_df["fill_rate"].min() if len(post_df) > 0 else np.nan,
        "final_cost": model.compute_total_cost(),
        "ttr": model.compute_time_to_recover(target_frac=0.95),
    }


def run_lead_time_test(lt_increase, strategy_config, seed):
    """Run lead_time_surge scenario with explicit lead time increase."""
    random.seed(seed)
    np.random.seed(seed)
    
    from model import MultiTierModel
    
    assumptions = {
        "n_suppliers": 4, "n_plants": 2, "n_dcs": 2, "n_retailers": 4,
        "base_stock": {"supplier": 60, "plant": 50, "dc": 30, "retailer": 20},
        "capacity": {"supplier": 50, "plant": 60, "dc": 0, "retailer": 0},
        "lead_time": {"supplier": 3, "plant": 2, "dc": 2, "retailer": 1},
        "capacity_loss_frac": 0.1, "recovery_duration": 5,
        "holding_cost": 1.0, "backlog_cost": 5.0,
        "retailer_demand_mean": 7.0
    }
    
    model = MultiTierModel(
        assumptions=assumptions,
        seed=seed,
        strategies=strategy_config.copy(),
        disruption_at_step=DISRUPTION_AT_STEP,
        scenario="lead_time_surge",
        n_steps=N_STEPS
    )
    
    increase = lt_increase
    def patched_trigger():
        if (not model.disruption_done) and (model.time == model.disruption_at_step):
            candidates = [a for a in model.all_agents if a.tier in ("plant", "dc")]
            if candidates:
                victim = random.choice(candidates)
                tier = victim.tier
                affected = [a for a in model.all_agents if a.tier == tier]
                for agent in affected:
                    agent._orig_lead_time = agent.lead_time
                    agent.lead_time += increase
                    agent.is_disrupted = True
                    agent.recovery_timer = model.recovery_duration
            model.disruption_done = True
            model.disruption_step = model.time
    
    model._maybe_trigger_disruption = patched_trigger
    model.scenario = "capacity_loss"
    
    for _ in range(N_STEPS):
        model.step()
    
    df = model.datacollector.get_model_vars_dataframe()
    post_df = df[df.index >= DISRUPTION_AT_STEP]
    
    return {
        "post_fill": post_df["fill_rate"].mean() if len(post_df) > 0 else np.nan,
        "min_fill": post_df["fill_rate"].min() if len(post_df) > 0 else np.nan,
        "final_cost": model.compute_total_cost(),
        "ttr": model.compute_time_to_recover(target_frac=0.95),
    }


def run_recovery_duration_test(recovery_dur, scenario, strategy_config, seed):
    """Run specified scenario with explicit recovery duration."""
    random.seed(seed)
    np.random.seed(seed)
    
    from model import MultiTierModel
    
    # Use stronger disruptions to see effect of recovery duration
    assumptions = {
        "n_suppliers": 4, "n_plants": 2, "n_dcs": 2, "n_retailers": 4,
        "base_stock": {"supplier": 60, "plant": 50, "dc": 30, "retailer": 20},
        "capacity": {"supplier": 50, "plant": 60, "dc": 0, "retailer": 0},
        "lead_time": {"supplier": 3, "plant": 2, "dc": 2, "retailer": 1},
        "capacity_loss_frac": 0.5,  # Strong disruption
        "recovery_duration": recovery_dur,
        "holding_cost": 1.0, "backlog_cost": 5.0,
        "retailer_demand_mean": 7.0
    }
    
    model = MultiTierModel(
        assumptions=assumptions,
        seed=seed,
        strategies=strategy_config.copy(),
        disruption_at_step=DISRUPTION_AT_STEP,
        scenario=scenario,
        n_steps=N_STEPS
    )
    
    # Override recovery_duration explicitly
    model.recovery_duration = recovery_dur
    
    for _ in range(N_STEPS):
        model.step()
    
    df = model.datacollector.get_model_vars_dataframe()
    post_df = df[df.index >= DISRUPTION_AT_STEP]
    
    return {
        "post_fill": post_df["fill_rate"].mean() if len(post_df) > 0 else np.nan,
        "min_fill": post_df["fill_rate"].min() if len(post_df) > 0 else np.nan,
        "final_cost": model.compute_total_cost(),
        "ttr": model.compute_time_to_recover(target_frac=0.95),
    }


# ============================================
# MAIN EXECUTION
# ============================================
print("="*60)
print("SENSITIVITY ANALYSIS - FIXED VERSION")
print("="*60)

all_results = []

# 1. Demand Spike Factor
print("\n[1/3] Demand Spike Factor...")
for value in [2, 3, 5, 7]:
    print(f"  Testing demand_spike = {value}x")
    for strategy_name, strategy_config in STRATEGY_CONFIGS.items():
        metrics = run_demand_spike_test(value, strategy_config, SEED)
        all_results.append({
            "param_name": "demand_spike_factor",
            "param_value": value,
            "strategy": strategy_name,
            **metrics
        })

# 2. Lead Time Surge
print("\n[2/3] Lead Time Surge...")
for value in [4, 8, 12, 16]:
    print(f"  Testing lead_time_surge = +{value}")
    for strategy_name, strategy_config in STRATEGY_CONFIGS.items():
        metrics = run_lead_time_test(value, strategy_config, SEED)
        all_results.append({
            "param_name": "lead_time_surge",
            "param_value": value,
            "strategy": strategy_name,
            **metrics
        })

# 3. Recovery Duration (only on demand_spike - most informative)
print("\n[3/3] Recovery Duration (demand_spike scenario)...")
for value in [3, 5, 7, 10]:
    print(f"  Testing recovery_duration = {value}")
    for strategy_name, strategy_config in STRATEGY_CONFIGS.items():
        metrics = run_recovery_duration_test(value, "demand_spike", strategy_config, SEED)
        all_results.append({
            "param_name": "recovery_duration",
            "param_value": value,
            "strategy": strategy_name,
            **metrics
        })

# Save results
results_df = pd.DataFrame(all_results)
results_df.to_csv(f"{OUTPUT_DIR}/sensitivity_raw.csv", index=False)
print(f"\n✓ Saved: {OUTPUT_DIR}/sensitivity_raw.csv ({len(results_df)} rows)")

# ============================================
# GENERATE ONLY 2 COMBINED FIGURES
# ============================================
print("\n" + "="*60)
print("GENERATING FIGURES")
print("="*60)

# Figure 1: Fill Rate (2x2 grid)
fig1, axes1 = plt.subplots(1, 3, figsize=(15, 5))

params = [
    ("demand_spike_factor", "Demand Spike Factor"),
    ("lead_time_surge", "Lead Time Increase (periods)"),
    ("recovery_duration", "Recovery Duration (periods)"),
]

for idx, (param_name, param_label) in enumerate(params):
    ax = axes1[idx]
    subset = results_df[results_df["param_name"] == param_name]
    
    for strategy in STRATEGY_CONFIGS.keys():
        data = subset[subset["strategy"] == strategy].sort_values("param_value")
        if len(data) > 0:
            ax.plot(data["param_value"], data["post_fill"],
                   marker='o', linewidth=2.5, markersize=8,
                   color=STRATEGY_COLORS[strategy],
                   label=STRATEGY_LABELS[strategy])
    
    ax.set_xlabel(param_label, fontsize=11, fontweight='bold')
    ax.set_ylabel("Average Fill Rate", fontsize=11)
    ax.set_ylim(0.4, 1.05)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add subplot label
    ax.text(0.02, 0.98, f"({chr(97+idx)})", transform=ax.transAxes, 
            fontsize=12, fontweight='bold', va='top')

# Single legend at bottom
handles, labels = axes1[0].get_legend_handles_labels()
fig1.legend(handles, labels, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.02), frameon=True)

fig1.suptitle('Sensitivity Analysis: Average Fill Rate', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
fig1.savefig(f"{OUTPUT_DIR}/sensitivity_fill_rate.png", dpi=300, bbox_inches='tight', facecolor='white')
fig1.savefig(f"{OUTPUT_DIR}/sensitivity_fill_rate.pdf", bbox_inches='tight', facecolor='white')
plt.close(fig1)
print("  ✓ sensitivity_fill_rate.png/pdf")

# Figure 2: Total Cost (2x2 grid)
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))

for idx, (param_name, param_label) in enumerate(params):
    ax = axes2[idx]
    subset = results_df[results_df["param_name"] == param_name]
    
    for strategy in STRATEGY_CONFIGS.keys():
        data = subset[subset["strategy"] == strategy].sort_values("param_value")
        if len(data) > 0:
            ax.plot(data["param_value"], data["final_cost"] / 1000,  # Convert to thousands
                   marker='s', linewidth=2.5, markersize=8,
                   color=STRATEGY_COLORS[strategy],
                   label=STRATEGY_LABELS[strategy])
    
    ax.set_xlabel(param_label, fontsize=11, fontweight='bold')
    ax.set_ylabel("Total Cost (×1000)", fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.text(0.02, 0.98, f"({chr(97+idx)})", transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='top')

handles, labels = axes2[0].get_legend_handles_labels()
fig2.legend(handles, labels, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.02), frameon=True)

fig2.suptitle('Sensitivity Analysis: Total Cost', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
fig2.savefig(f"{OUTPUT_DIR}/sensitivity_cost.png", dpi=300, bbox_inches='tight', facecolor='white')
fig2.savefig(f"{OUTPUT_DIR}/sensitivity_cost.pdf", bbox_inches='tight', facecolor='white')
plt.close(fig2)
print("  ✓ sensitivity_cost.png/pdf")

print("\n" + "="*60)
print("COMPLETE - Generated 2 combined figures")
print("="*60)
print(f"Output: {OUTPUT_DIR}/")
print("  - sensitivity_raw.csv")
print("  - sensitivity_fill_rate.png/pdf")
print("  - sensitivity_cost.png/pdf")

SENSITIVITY ANALYSIS - FIXED VERSION

[1/3] Demand Spike Factor...
  Testing demand_spike = 2x
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
  Testing demand_spike = 3x
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
  Testing demand_spike = 5x
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[Recovery] t=14 all retailers restored (demand: 7.0)
[